# DeepMath Dataset Builder: Symbolic Mathematical Pretraining Curation

This notebook streams high-density mathematical reasoning corpora (such as  , , and ), runs the **Discrete Evidence Engine (DEE)** for mathematical formula integrity and LaTeX validation, and outputs compressed Snappy Parquet shards partitioned into **SmolLM2 developmental stages** (Stages 2, 3, and 4).

In [ ]:
# 1. Install prerequisites
!pip install -q datasets pyarrow pandas

In [ ]:
# 2. Stage DEE standalone binary
import os, shutil, glob
candidates = glob.glob('/kaggle/input/**/dee', recursive=True) + glob.glob('../input/**/dee', recursive=True) + glob.glob('./dee')
dee_bin = '/kaggle/working/dee'
if candidates:
    src = candidates[0]
    if os.path.abspath(src) != os.path.abspath(dee_bin):
        shutil.copy2(src, dee_bin)
    os.chmod(dee_bin, 0o755)
    print(f'Staged DEE binary: {dee_bin}')
else:
    raise FileNotFoundError('DEE binary not found in inputs')
!{dee_bin} --help

In [ ]:
# 3. Locate builder and execute DeepMath streaming
import os, glob
candidates = glob.glob('/kaggle/input/**/deepmath_builder.py', recursive=True) + glob.glob('./deepmath_builder.py')
script = candidates[0] if candidates else 'deepmath_builder.py'
print(f'Using builder: {script}')

# Stream FineMath-4Plus with 18.5GB disk-space guard
!python3 {script} --dataset HuggingFaceTB/finemath --subset finemath-4plus --chunk-size 5000 --output-dir /kaggle/working/deepmath --dee-binary {dee_bin}

In [ ]:
# 4. Summary of Generated DeepMath Curriculum Partitions
from pathlib import Path
import pandas as pd

parquet_root = Path('/kaggle/working/deepmath/parquet_shards')
print('=' * 75)
print('DEEPMATH PARQUET CURRICULUM READY')
print('=' * 75)
for sdir in sorted(parquet_root.iterdir()):
    if sdir.is_dir():
        shards = list(sdir.glob('*.parquet'))
        total_mb = sum(f.stat().st_size for f in shards) / (1024*1024)
        print(f'{sdir.name:<26}: {len(shards):>4} shards | {total_mb:>8.2f} MB')
print('=' * 75)